# Splane DLPFC(1/2): Identify uniform spatial domain

## Data preprocessing

In [ ]:
from SPACEL.setting import set_environ_seed
seed = 1
set_environ_seed(seed)
from SPACEL import Splane
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib
import anndata
import glob
import os
from natsort import natsorted

Setting environment seed: 11


Here, we will incorporate the cell type composition predicted by **Spoint** into the spatial anndata object for subsequent spatial domain identification in **Splane** using the `add_cell_type_composition` function. This function takes a DataFrame containing the cell type composition matrix as input for spot-based spatial transcriptomic data or a series of cell type annotations as input for single-cell resolution spatial transcriptomic data.

In [ ]:
# Path to the directory
data_dir = './sim'
slices = ['151673', '100', '80', '60', '40', '20']

# Find all .h5ad files
h5ad_files = natsorted(glob.glob(os.path.join(data_dir, '*.h5')))
h5ad_files.reverse()
# h5ad_files

adata_list = []
# Read all .h5ad files into a list of AnnData objects
for f in h5ad_files:
    ad = sc.read(f)
    adata_list.append(ad)

In [3]:
adata_list[0].obs['original_clusters'].unique()

['3', '1', '0', '5', '6', '2', '4']
Categories (7, object): ['0', '1', '2', '3', '4', '5', '6']

In [4]:
# Collect all unique cell types across all .h5ad files
all_cell_types = set()
for adata in adata_list:
    all_cell_types.update(adata.obs['original_clusters'].unique().tolist())

all_cell_types = sorted(list(all_cell_types)) 
all_cell_types

['0', '1', '2', '3', '4', '5', '6']

In [5]:
for i, adata in enumerate(adata_list):    
    # Add cell type composition
    Splane.utils.add_cell_type_composition(adata, celltype_anno=adata.obs['original_clusters'])
    # Add missing cell type columns as 0s
    for ct in all_cell_types:
        if ct not in adata.obs.columns:
            adata.obs[ct] = 0.0
            
    # Reorder columns consistently
    ordered_cols = [c for c in adata.obs.columns if c not in all_cell_types] + all_cell_types
    adata.obs = adata.obs[ordered_cols]
    adata.obs['slice_id'] = slices[i]
    adata.uns['celltypes'] = np.array(all_cell_types, dtype=object)

In [6]:
adata_list[1].obs

,in_tissue,array_row,array_col,original_clusters,SpotIndex,0,1,2,3,4,5,6,slice_id
AAACAAGTATCTCCCA-1,1,50,102,3,0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,100
AAACAATCTACTAGCA-1,1,3,43,1,1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,100
AAACACCAATAACTGC-1,1,59,19,0,2,1.0,0.0,0.0,0.0,0.0,0.0,0.0,100
AAACAGAGCGACTCCT-1,1,14,94,3,3,0.0,0.0,0.0,1.0,0.0,0.0,0.0,100
AAACAGCTTTCAGAAG-1,1,43,9,5,4,0.0,0.0,0.0,0.0,0.0,1.0,0.0,100
...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTGTTGTGTGTCAAGA-1,1,31,77,5,3605,0.0,0.0,0.0,0.0,0.0,1.0,0.0,100
TTGTTTCACATCCAGG-1,1,58,42,0,3606,1.0,0.0,0.0,0.0,0.0,0.0,0.0,100
TTGTTTCATTAGTCTA-1,1,60,30,0,3607,1.0,0.0,0.0,0.0,0.0,0.0,0.0,100
TTGTTTCCATACAACT-1,1,45,27,6,3608,0.0,0.0,0.0,0.0,0.0,0.0,1.0,100


## Training Splane model

In this step, we initialize the Splane model by ``Splane.init_model(...)`` using the anndata object list as input. The ``n_clusters`` parameter determines the number of spatial domains to be identified. The ``k`` parameter controls the degree of neighbors considered in the model, with a larger ``k`` value resulting in more emphasis on global structure rather than local structure. The ``gnn_dropout`` parameter influences the level of smoothness in the model’s predictions, with a higher ``gnn_dropout`` value resulting in a smoother output that accommodates the sparsity of the spatial transcriptomics data. 

We train the model by ``splane.train(...)`` to obtain latent feature of each spots/cells. The parameter ``d_l`` affects the level of batch effect correction between slices. By default, ``d_l`` is ``0.2`` for spatial transcriptomics data with single cell resolution.

Then, we can identify the spatial domain to which each spot/cell belongs by ``splane.identify_spatial_domain(...)``. By default, the results will be saved in ``spatial_domain`` column in ``.obs``. If the key parameter is provided, the results will be saved in ``.obs[key]``.

In [ ]:
splane_model = Splane.init_model(adata_list, n_clusters=7,use_gpu=False,n_neighbors=25, gnn_dropout=0.5, seed=seed)
splane_model.train(d_l=0.2)
splane_model.identify_spatial_domain()
    
sc.concat(adata_list).write(f'./{seed}/simData_151673_20_Splane.h5ad')    

Setting global seed: 11
Calculating cell type weights...


/hpc/group/hickeylab/fl135/miniconda3/envs/SPACEL/lib/python3.8/site-packages/SPACEL/Splane/utils.py:76: RuntimeWarning: invalid value encountered in double_scalars
  return len(x) / W * inum / z2ss
/hpc/group/hickeylab/fl135/miniconda3/envs/SPACEL/lib/python3.8/site-packages/SPACEL/Splane/pygcn_utils.py:11: RuntimeWarning: divide by zero encountered in power
  d = sp.diags(np.power(np.array(adj.sum(1)), -1).flatten(), 0)


Generating GNN inputs...
Calculating largest eigenvalue of normalized graph Laplacian...
Calculating Chebyshev polynomials up to order 2...


/hpc/group/hickeylab/fl135/miniconda3/envs/SPACEL/lib/python3.8/site-packages/SPACEL/Splane/pygcn_utils.py:57: UserWarning: torch.sparse.SparseTensor(indices, values, shape, *, device=) is deprecated.  Please use torch.sparse_coo_tensor(indices, values, shape, dtype=, device=). (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:641.)
  return torch.sparse.FloatTensor(indices, values, shape)


Setting the weight of similarity loss to 1.498


The best epoch 250 total loss=-2.281 g loss=-1.922 d loss=1.794 d acc=0.250 simi loss=-0.982 db loss=0.409: 100%|██████████| 300/300 [06:21<00:00,  1.27s/it]
/hpc/home/fl135/.local/lib/python3.8/site-packages/anndata/_core/anndata.py:1838: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [8]:
adata_list[0]

AnnData object with n_obs × n_vars = 3611 × 33538
    obs: 'in_tissue', 'array_row', 'array_col', 'original_clusters', 'SpotIndex', '0', '1', '2', '3', '4', '5', '6', 'slice_id', 'spatial_domain'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'spatial', 'celltypes', 'spatial_neighbors', 'spatial_domain_colors'
    obsm: 'spatial'
    obsp: 'spatial_connectivities', 'spatial_distances'